# H1 volume and Brief A/B exploration · Group 10

This notebook combines source profiling, booking loading and cleaning with the supporting H1 growth and Brief A/B analysis. The primary growth measure counts active booking-test lines by scheduled date (`slot_at`); Brief C is the final board-facing question in Notebook 02.

Run top to bottom from the repository root or the `notebooks/` folder. Generated row-level clean files remain local and are ignored by Git.

## CONFIG

In [ ]:
import sys, pathlib, time
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "notebooks" / "_scripts"))
import numpy as np
import pandas as pd
from kalpa import loaders, models, checks, logs
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
CONFIG = {
    "brief": "Brief C",
    "group": "10",
    "raw": ROOT / "data" / "raw",
    "clean": ROOT / "data" / "clean",
    "period_this": ("2026-01-01", "2026-06-30"),
    "period_last": ("2025-01-01", "2025-06-30"),
    "seed": 20261019,
    "workers": 2,             # Keep the full-pack load within Codespace memory.
}
CONFIG["clean"].mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(CONFIG["seed"])
print(pd.__version__, np.__version__)

## 1 · MAP: what is in the pack

The pack is an export of six source systems. Each family has its own shape. The first move is to list every file with its size and family, so you know what you are dealing with before you read a byte of it.

In [ ]:
files = loaders.list_pack(CONFIG["raw"])
print(f"{len(files)} files, {files['bytes'].sum()/1e6:.1f} MB")
files.groupby("family").agg(files=("path", "count"), mb=("bytes", lambda b: round(b.sum()/1e6, 1))).sort_values("mb", ascending=False)

### CHECK

In [ ]:
import json, hashlib
manifest = json.load(open(CONFIG["raw"] / "MANIFEST.json"))
listed = {f["path"]: f for f in manifest["files"]}
missing = [p for p in listed if not (CONFIG["raw"] / p).exists()]
checks.check("every file in MANIFEST.json is present", len(missing) == 0, f"{len(missing)} missing")
# spot-check three hashes rather than all 300: enough to know the download is intact
sample = list(listed)[:: max(1, len(listed)//3)][:3]
ok = all(hashlib.sha256((CONFIG["raw"] / p).read_bytes()).hexdigest() == listed[p]["sha256"] for p in sample)
checks.check("spot-checked SHA-256 match the manifest", ok, sample)

## 2 · DO: read one legacy file the naive way, and watch it go wrong

A legacy monthly export is Latin-1, dates are day first, and the vendor appended files so some carry the header twice. Read one naively first. The failure is the lesson.

In [ ]:
legacy_files = sorted((CONFIG["raw"] / "bookings" / "legacy").rglob("*.csv.gz"))
biggest = lambda paths: max(paths, key=lambda p: p.stat().st_size)   # the largest file of a family is the honest sample; the smallest is often an edge
one = biggest(legacy_files)
print(one.name)
try:
    naive = pd.read_csv(one)           # default encoding is UTF-8
    print("read without error:", naive.shape)
except UnicodeDecodeError as err:
    print("UnicodeDecodeError:", str(err)[:120])

In [ ]:
naive = pd.read_csv(one, encoding="latin-1")
print(naive.dtypes)
print()
print("rows whose booking_id is literally the header:", (naive["booking_id"] == "booking_id").sum())
print("slot_at parsed as:", type(naive["slot_at"].iloc[0]).__name__, "->", naive["slot_at"].iloc[0])

> **Interview angle.** Why did `pd.read_csv` fail on the first attempt, and why would guessing the encoding be a mistake?
>
> *What a good answer covers:* The file is Latin-1; UTF-8 is the default. Guessing by trying encodings until one works can silently mangle accented characters into different, valid characters. Find the encoding from the source system or the vendor and state it in the loader.

## 3 · DO: the proper reader

`loaders.read_legacy_file` reads with the right encoding, drops repeated header lines and counts them, parses the day-first dates with an explicit format, strips the ids, and declares dtypes so a 200,000-row file costs a tenth of the memory it would as plain objects.

In [ ]:
t0 = time.time()
df = loaders.read_legacy_file(one)
print(f"{len(df):,} booking-test rows in {time.time()-t0:.2f}s; repeated header lines dropped: {df.attrs['repeated_header_lines']}")
print(df.dtypes)
print(f"memory: {df.memory_usage(deep=True).sum()/1e6:.1f} MB  (naive object frame: {naive.memory_usage(deep=True).sum()/1e6:.1f} MB)")
df.head(3)

> **Interview angle.** What does declaring `dtype={'city': 'category'}` change, and when is `category` the wrong choice?
>
> *What a good answer covers:* A category stores each distinct value once and an integer code per row, so memory drops and groupby gets faster. It is wrong for high-cardinality columns such as ids, where the codes table is as large as the data, and it needs care on concat when the two frames have different category sets.

## 4 · DO: profile every family with the same function

One function, applied to every table family: rows, distinct keys, null rate per column, min and max date, and the share of ids matching the expected format. Run it on one file per family here; notebook 02 runs the full load.

In [ ]:
def profile(df: pd.DataFrame, name: str, key: str | None = None) -> pd.DataFrame:
    rows = []
    for col in df.columns:
        s = df[col]
        rows.append({"table": name, "column": col, "dtype": str(s.dtype), "null_rate": round(float(s.isna().mean()), 4),
                     "distinct": int(s.nunique(dropna=True)),
                     "min": s.min() if pd.api.types.is_numeric_dtype(s) or pd.api.types.is_datetime64_any_dtype(s) else "",
                     "max": s.max() if pd.api.types.is_numeric_dtype(s) or pd.api.types.is_datetime64_any_dtype(s) else ""})
    out = pd.DataFrame(rows)
    if key is not None:
        print(f"{name}: {len(df):,} rows, {df[key].nunique():,} distinct {key}  ->  {len(df)/max(1, df[key].nunique()):.2f} rows per key")
    return out

prof = [profile(df, "bookings_legacy", "booking_id")]
nova_files = sorted((CONFIG["raw"] / "bookings" / "nova").rglob("*.jsonl.gz"))
if nova_files:
    nv = loaders.read_nova_file(biggest(nova_files))
    prof.append(profile(nv, "bookings_nova", "booking_id"))
coll = loaders.read_simple_csv(biggest(list((CONFIG["raw"] / "collections").glob("*.csv.gz"))), parse_dates=["collected_at"])
prof.append(profile(coll, "collections", "booking_id"))
res = loaders.read_simple_csv(biggest(list((CONFIG["raw"] / "results").rglob("*.csv.gz"))))
prof.append(profile(res, "results", "booking_id"))
inv = loaders.read_simple_csv(biggest(list((CONFIG["raw"] / "invoices").glob("*.csv.gz"))), dtype={"amount": "string"})
prof.append(profile(inv, "invoices", "invoice_id"))
pay = loaders.read_simple_csv(biggest(list((CONFIG["raw"] / "payments").glob("gateway_*.csv.gz"))))
prof.append(profile(pay, "payments", "payment_id"))
pat = pd.read_csv(CONFIG["raw"] / "patients" / "patients.csv.gz")
prof.append(profile(pat, "patients", "patient_id"))
profile_columns = pd.concat(prof, ignore_index=True)
profile_columns

### SEE: the things a profile shows that a head() does not

- `rows per key` above 1.0 on a bookings file means the grain is booking-test, not booking. Count bookings with `nunique`, never with `len`.
- An `amount` column that profiles as `string` holds thousands separators. Sum it and pandas raises, or worse, concatenates.
- A `created_at_epoch` column that profiles as integer with a min near 1.7 billion is Unix seconds.
- A null rate of 0.003 on `test_code` is small until you remember that every one of those rows is a test you cannot price.

In [ ]:
id_ok = df["booking_id"].str.match(loaders.ID_PATTERNS["legacy"].pattern)
checks.check("every legacy booking id matches BK-CCC-YYYYMM-NNNNNN after strip", bool(id_ok.all()), f"{(~id_ok).sum()} do not")
checks.check("legacy slot_at parsed for every row", bool(df["slot_at"].notna().all()), f"{df['slot_at'].isna().sum()} NaT")
checks.check("invoice amount is text with separators (so we know to parse it)", pd.api.types.is_string_dtype(inv["amount"]), inv["amount"].iloc[0])
checks.check("payments amount is in paise (integer, large)", pd.api.types.is_integer_dtype(pay["amount_paise"]) and pay["amount_paise"].median() > 1000, pay["amount_paise"].median())

## 5 · DO: the identifier audit

Every table that should join on an id gets its id format checked against a pattern. Mismatches are a list of things to decide about in this notebook's cleaning section.

In [ ]:
audit = {
    "legacy booking_id (raw, before strip)": naive["booking_id"].astype("string").str.match(r"^BK-[A-Z]{3}-\d{6}-\d{6}$").mean(),
    "collections booking_id": coll["booking_id"].astype("string").str.match(r"^(BK-[A-Z]{3}-\d{6}-\d{6}|nova_[a-z]{3}_\d{7})$").mean(),
    "results booking_id": res["booking_id"].astype("string").str.match(r"^(BK-[A-Z]{3}-\d{6}-\d{6}|nova_[a-z]{3}_\d{7})$").mean(),
    "invoice_id": inv["invoice_id"].astype("string").str.match(r"^INV-\d{4}-\d{6}$").mean(),
    "payment notes_invoice_ref, exact format": pay["notes_invoice_ref"].astype("string").str.match(r"^INV-\d{4}-\d{6}$").mean(),
    "payment notes_invoice_ref, after normalisation": loaders.normalise_invoice_ref(pay["notes_invoice_ref"]).str.match(r"^INV-\d{4}-\d{6}$").mean(),
}
pd.Series(audit, name="share_matching_pattern").round(4).to_frame()

> **Interview angle.** A payment reference is 'inv-2026-000123 ' with a trailing space. Is normalising it a cleaning decision or a parsing decision, and who should be able to audit it?
>
> *What a good answer covers:* It is a decision: you are asserting that this string identifies that invoice. It is reversible and low-risk, so it belongs in the decisions log with its rule and the count of rows it changed. A transposed digit is different: it cannot be repaired without guessing, so it stays unmatched and is listed.

## 6 · SUM: write the profile out and log the first decisions

In [ ]:
files.to_csv(CONFIG["clean"] / "profile_files.csv", index=False)
profile_columns.to_csv(CONFIG["clean"] / "profile_columns.csv", index=False)
print("wrote profile_files.csv and profile_columns.csv")
# Example of recording a decision the moment it is made. Uncomment and fill in when you have decided.
# logs.decision("bookings_legacy", "header line repeated mid-file", "drop", "an append artefact, not data",
#               rows_affected=df.attrs["repeated_header_lines"], effect_on_headline="none (not rows)", owner="Rishabh", n=1)
checks.summary()

## Before you move on

- Which table is the source of truth for a count of tests, and why that one? Write it in `docs/approach-log.md` row 1.
- Which join will be hardest, and what did the id audit tell you about it?
- What did you expect the row counts to be before you looked? How far off were you?

## Short answers

1. **Test-count source:** Use normalized booking-test rows, count nonblank active tests by scheduled date, and exclude superseded reschedules. Collections and results are later events, not the plan's scheduled-volume denominator.
2. **Hardest join:** Gateway payment reference to invoice ID. Match exact IDs, then hyphen-stripped IDs only when unique; preserve other references as unmatched instead of guessing.
3. **Expected row counts:** No numerical pre-profile estimate was recorded, so we cannot honestly report forecast error. The later load observed 1,315,183 booking-test rows across 567,986 booking IDs before validation.

## Load and validate booking sources · Group 10

**Produces:** `data/clean/bookings_all.csv.gz` (one row per booking-test, both systems, normalised), `data/clean/bookings.csv.gz` (one row per booking), `data/clean/rejects_bookings.csv` (every rejected record with its reason).

**Needs:** notebook 01 run; the pack in `data/raw/`.

Two ideas in this notebook. First, the pack is a few hundred files, so the unit of work is a file and a process pool reads them side by side. Second, nothing enters the analysis without passing a model that says what a booking must look like; what fails is kept, counted and listed, so input equals accepted plus rejected.

## CONFIG

In [ ]:
import sys, pathlib, time
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "notebooks" / "_scripts"))
import numpy as np
import pandas as pd
from kalpa import loaders, models, checks, logs
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
CONFIG = {
    "brief": "Brief C",
    "group": "10",
    "raw": ROOT / "data" / "raw",
    "clean": ROOT / "data" / "clean",
    "period_this": ("2026-01-01", "2026-06-30"),
    "period_last": ("2025-01-01", "2025-06-30"),
    "seed": 20261019,
    "workers": 2,             # Keep the full-pack load within Codespace memory.
}
CONFIG["clean"].mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(CONFIG["seed"])
print(pd.__version__, np.__version__)

## 1 · MAP: two systems, one grain

Legacy exports are one row per booking-test. Nova exports are one JSON object per booking with the tests nested. Both are brought to booking-test grain by the loaders, then rolled up to booking grain here. The picture:

```
legacy/*.csv.gz  ─┐                           ┌─ bookings_all (booking-test grain)
                  ├─ process pool ─ concat ───┤
nova/*.jsonl.gz  ─┘        │                  └─ bookings (booking grain, n_tests, amount)
                           └─ pydantic validation ─ rejects_bookings
```

## 2 · DO: sequential against parallel, timed, so the choice is a measured one

In [ ]:
legacy_files = sorted((CONFIG["raw"] / "bookings" / "legacy").rglob("*.csv.gz"))
nova_files = sorted((CONFIG["raw"] / "bookings" / "nova").rglob("*.jsonl.gz"))
print(len(legacy_files), "legacy files,", len(nova_files), "nova files")

t0 = time.time()
seq = pd.concat([loaders.read_legacy_file(p) for p in legacy_files[:12]], ignore_index=True)
t_seq = time.time() - t0
t0 = time.time()
par = loaders.read_many(legacy_files[:12], "legacy", workers=CONFIG["workers"])
t_par = time.time() - t0
print(f"12 files: sequential {t_seq:.1f}s, process pool {t_par:.1f}s, same rows: {len(seq) == len(par)}")

> **Interview angle.** Why a process pool and not a thread pool here? When would threads be the right choice?
>
> *What a good answer covers:* Decompressing gzip, parsing CSV and parsing JSON are CPU-bound Python work, and the global interpreter lock lets only one thread run Python bytecode at a time, so threads would queue. Processes each have their own interpreter. Threads win when the work waits on I/O, such as network calls or disk reads of already-parsed data, and when the data to pass between workers is large, since processes copy it.

## 3 · DO: the full load

In [ ]:
t0 = time.time()
legacy = loaders.read_many(legacy_files, "legacy", workers=CONFIG["workers"])
print(f"legacy: {len(legacy):,} booking-test rows from {len(legacy_files)} files in {time.time()-t0:.1f}s; repeated headers dropped: {legacy.attrs['repeated_header_lines']}")
t0 = time.time()
nova = loaders.read_many(nova_files, "nova", workers=CONFIG["workers"]) if nova_files else pd.DataFrame()
print(f"nova: {len(nova):,} booking-test rows from {len(nova_files)} files in {time.time()-t0:.1f}s")

In [ ]:
# Concatenating two frames whose columns differ: name the difference before you union, so a silent NaN column cannot surprise you later
only_legacy = set(legacy.columns) - set(nova.columns)
only_nova = set(nova.columns) - set(legacy.columns)
print("only in legacy:", only_legacy, "| only in nova:", only_nova)
for c in only_nova: legacy[c] = pd.NA
for c in only_legacy: nova[c] = pd.NA
all_rows = pd.concat([legacy, nova[legacy.columns]], ignore_index=True)
all_rows["city"] = all_rows["city"].astype("string").replace({"Bangalore": "Bengaluru"})
print(f"{len(all_rows):,} booking-test rows; {all_rows['booking_id'].nunique():,} distinct bookings")
all_rows.groupby("system", observed=True).agg(rows=("booking_id", "size"), bookings=("booking_id", "nunique"), first=("slot_at", "min"), last=("slot_at", "max"))

### SEE: the grain trap, in numbers

The table above is the whole of Brief B in one line: rows per booking differ by system, so a row count across the migration is not a booking count. Nova also carries rescheduled records. Decide what a booking is, write the rule, and apply it in the cleaning section below.

## 4 · DO: validate every booking with a model

`models.Booking` says what one visit must look like. Build one record per booking from the rows, validate each, and keep the rejects with reasons. This is slower than a vectorised check and that is the point of doing it once at the boundary: the model is the contract, and the rejects file is the evidence that nothing slipped past it.

In [ ]:
def records_from_rows(rows: pd.DataFrame):
    """Group booking-test rows into one dict per booking, the shape the model expects.

    Sorted once, then sliced by the boundaries np.unique returns: a few seconds for a million rows,
    where a groupby.apply with a Python lambda would take minutes. Same result, measured difference."""
    cols = ["booking_id", "system", "created_at", "slot_at", "city", "clinic_id", "channel", "source", "patient_id", "home_fee", "rescheduled_from", "status_at_export"]
    r = rows.sort_values("booking_id", kind="stable").reset_index(drop=True)
    ids = r["booking_id"].to_numpy()
    _, starts = np.unique(ids, return_index=True)
    ends = np.append(starts[1:], len(r))
    meta = r[cols].to_dict("records")
    codes = r["test_code"].astype("string").fillna("").to_numpy()
    prices = r["test_price"].fillna(0).astype(int).to_numpy()
    for a, b in zip(starts, ends):
        rec = {k: (None if (v is pd.NA or (isinstance(v, float) and v != v)) else v) for k, v in meta[a].items()}
        rec["tests"] = [{"code": c, "price": int(p)} for c, p in zip(codes[a:b], prices[a:b])]
        yield rec

t0 = time.time()
accepted, rejects = models.validate_many(models.Booking, records_from_rows(all_rows))
print(f"validated {len(accepted)+len(rejects):,} bookings in {time.time()-t0:.0f}s (Python-level validation, once, at the boundary): accepted {len(accepted):,}, rejected {len(rejects):,}")
checks.reconciles("booking validation", all_rows["booking_id"].nunique(), len(accepted), len(rejects))

In [ ]:
rej = pd.DataFrame([{"booking_id": r["record"].get("booking_id"), "system": r["record"].get("system"),
                     "reason": "; ".join(f"{f}: {m}" for f, m in r["errors"])} for r in rejects])
rej.to_csv(CONFIG["clean"] / "rejects_bookings.csv", index=False)
rej["reason"].str.split(":").str[0].value_counts() if len(rej) else print("no rejects")

> **Interview angle.** What does a pydantic model give you that a pandas `.str.match` does not, and what does it cost?
>
> *What a good answer covers:* A single, named contract for a record that normalises what is safe to normalise and refuses the rest with a per-field reason, reusable by the SQL loader, the API and the tests. It costs time per record (Python-level validation) and a second representation of the data, so you run it once at the boundary and vectorise everything after.

## 5 · DO: the booking-grain table

In [ ]:
bookings = pd.DataFrame([{**b.model_dump(exclude={"tests"}), "n_tests": b.n_tests, "amount_rs": b.amount_rs,
                          "has_package": any(t.code.startswith("PK") for t in b.tests)} for b in accepted])
for c in ("created_at", "slot_at"):
    bookings[c] = pd.to_datetime(bookings[c]).dt.tz_localize(None)
bookings["month"] = bookings["slot_at"].dt.to_period("M").astype(str)
bookings["is_reschedule"] = bookings["rescheduled_from"].notna()
print(bookings.shape)
bookings.groupby(["system", "is_reschedule"]).size()

In [ ]:
all_rows.to_csv(CONFIG["clean"] / "bookings_all.csv.gz", index=False, compression="gzip")
bookings.to_csv(CONFIG["clean"] / "bookings.csv.gz", index=False, compression="gzip")
checks.check("booking table has one row per booking id", bookings["booking_id"].is_unique, len(bookings))
checks.check("n_tests per booking is at least 1", bool((bookings["n_tests"] >= 1).all()))
checks.summary()

## 6 · SUM

You now hold every booking from both systems in one table at one grain, with every rejected record listed. Record in `docs/decisions-log.md`: the repeated-header count, the city spelling rule, the reject count and reasons. Record in `docs/approach-log.md`: why booking grain and not booking-test grain for the headline, and what the alternative would have changed.

## Before you move on

- What is a booking? One sentence. Does a rescheduled nova record count as a second booking?
- How many bookings did the validator reject, and could any of them have been real visits?
- If a reviewer re-ran this on a fresh Codespace, would they get the same reject count? Why?

## Short answers

1. **What is a booking?** One visit ID with one or more test lines. A Nova row marked rescheduled is superseded, not a second active visit for the volume measure.
2. **How many were rejected?** 3,107 of 567,986 booking records failed validation. Some could be real visits with incomplete or bad fields, so they remain listed for review rather than being silently deleted.
3. **Will a cold run agree?** With identical source files, code, schema, and validation rules, accepted/rejected counts should reproduce. Changed inputs, dependencies, or rules can change the count and must be logged.

## Clean and reconcile booking data · Group 10

**Produces:** `data/clean/bookings_clean.csv.gz`, `data/clean/collections_clean.csv.gz`, `data/clean/patients_clean.csv.gz`, `data/clean/invoices_clean.csv.gz`, `data/clean/payments_clean.csv.gz`, and rows in the decisions log for every rule applied.

**Needs:** notebook 02 run.

Every cleaning act here is a decision with a reason, a row count and an effect on the headline. Input equals clean plus rejected after every step. Joins are checked before and after on row count, and a join that grows is explained or reverted.

## CONFIG

In [ ]:
import sys, pathlib, time
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "notebooks" / "_scripts"))
import numpy as np
import pandas as pd
from kalpa import loaders, models, checks, logs
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
CONFIG = {
    "brief": "Brief C",
    "group": "10",
    "raw": ROOT / "data" / "raw",
    "clean": ROOT / "data" / "clean",
    "period_this": ("2026-01-01", "2026-06-30"),
    "period_last": ("2025-01-01", "2025-06-30"),
    "seed": 20261019,
    "workers": 2,             # Keep the full-pack load within Codespace memory.
}
CONFIG["clean"].mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(CONFIG["seed"])
print(pd.__version__, np.__version__)

In [ ]:
bookings = pd.read_csv(CONFIG["clean"] / "bookings.csv.gz", parse_dates=["created_at", "slot_at"], dtype={"booking_id": "string", "patient_id": "string", "rescheduled_from": "string", "status_at_export": "string"})
HEADLINE = lambda df: f"{len(df):,} bookings, {int(df['n_tests'].sum()):,} tests"
n0 = len(bookings)
print(HEADLINE(bookings))

## 1 · DO: the identity rule for a booking

Nova records a rescheduled visit as a new booking that points at the old one, and marks the old one `rescheduled`. Legacy overwrote the slot in place. If the question is "how many visits happened", a rescheduled pair is one visit. Write the rule, apply it, log it.

In [ ]:
# status_at_export is NA for every legacy row. A nullable boolean treats NA as False only if you say so; left alone,
# ~NA is NA and the legacy rows vanish from the filter. The trap is worth seeing once, so it is stated here.
is_superseded = bookings["status_at_export"].eq("rescheduled").fillna(False).astype(bool)
kept = bookings[~is_superseded].copy()
rejected = bookings[is_superseded]
checks.reconciles("drop superseded nova records", n0, len(kept), len(rejected))
print("after:", HEADLINE(kept))
# logs.decision("bookings", "nova writes a new record on reschedule and marks the old one", "drop the superseded original",
#               "one visit, one booking; legacy overwrote in place so this makes the systems comparable",
#               rows_affected=len(rejected), effect_on_headline=f"{n0:,} -> {len(kept):,} bookings", owner="Rishabh", n=2)
bookings = kept

> **Interview angle.** What is `pd.NA`, and what happens to `~series.eq('x')` when the series holds NA?
>
> *What a good answer covers:* pd.NA is the missing marker for nullable dtypes and propagates through comparisons, so eq gives NA, the negation stays NA, and a boolean filter drops those rows silently. fillna(False) before negating, or compare with notna() explicitly.

> **Interview angle.** Two systems record the same business event in different shapes. What is the general method for making them comparable, and what is the trap?
>
> *What a good answer covers:* Trace one real event through both systems by hand and write down how each represents it; then write a rule that maps both to one definition of the event; then apply it and check the count before and after. The trap is comparing totals without the rule and attributing the difference to the business.

## 2 · DO: collections, double scans, and the join that must not grow

In [ ]:
coll_files = sorted((CONFIG["raw"] / "collections").glob("*.csv.gz"))
coll = loaders.read_many(coll_files, "legacy", workers=1) if False else pd.concat([pd.read_csv(p, dtype={"booking_id": "string"}) for p in coll_files], ignore_index=True)
coll["collected_at"] = pd.to_datetime(coll["collected_at"], errors="coerce")
n_in = len(coll)
dups = coll.duplicated(subset=["booking_id", "status", "collected_at"], keep="first")
coll_clean = coll[~dups].copy()
checks.reconciles("collections exact duplicates (double scans)", n_in, len(coll_clean), int(dups.sum()))
checks.check("one collection row per booking", coll_clean["booking_id"].is_unique, f"{(~coll_clean['booking_id'].is_unique)} duplicates remain" if not coll_clean["booking_id"].is_unique else "")

In [ ]:
# LEFT join bookings to collections. The row count must not grow. validate= makes pandas raise rather than lie.
try:
    joined = bookings.merge(coll_clean[["booking_id", "status", "collected_at", "phlebotomist_id"]], on="booking_id", how="left", validate="one_to_one", indicator=True)
except pd.errors.MergeError as err:
    print("MergeError:", err)
    joined = bookings.merge(coll_clean.drop_duplicates("booking_id")[["booking_id", "status", "collected_at", "phlebotomist_id"]], on="booking_id", how="left", validate="one_to_one", indicator=True)
checks.join_did_not_fan_out("bookings LEFT JOIN collections", bookings, joined, "booking_id")
joined["_merge"].value_counts()

### SEE: what `left_only` means

A booking with no collection row is a cancellation (the phlebotomy app does not record cancelled bookings) or a booking in the future at export time. Decide which, and write it down. A row that is `left_only` and dated months ago is a question for the data team, and a caveat on your slide.

In [ ]:
joined["status"] = joined["status"].fillna("cancelled_or_missing")
skew = joined["collected_at"] < joined["slot_at"] - pd.Timedelta(days=1)
print("collected more than a day before the slot (clock skew?):", int(skew.sum()))
joined.loc[skew, "collected_at"] = pd.NaT       # keep the row, blank the timestamp, flag it
joined["flag_clock_skew"] = skew
bookings = joined.drop(columns="_merge")

> **Interview angle.** Explain `how='left'`, `validate='one_to_one'` and `indicator=True` to someone who has only used SQL.
>
> *What a good answer covers:* LEFT keeps every left row whether or not it matches, like SQL LEFT JOIN. validate asserts the key's cardinality on each side and raises if it is violated, which SQL does not do for you. indicator adds a column saying which side each row came from, which is the SQL anti-join check (WHERE right.key IS NULL) made explicit.

## 3 · DO: patients, and the duplicate that looks different

In [ ]:
pat = pd.read_csv(CONFIG["raw"] / "patients" / "patients.csv.gz", dtype="string")
pat["phone_norm"] = loaders.normalise_phone(pat["phone"])
pat["name_norm"] = pat["patient_name"].str.upper().str.replace(r"\s+", " ", regex=True).str.strip()
grp = pat.groupby(["phone_norm", "name_norm", "dob"]).ngroup()
pat["person_key"] = grp
dup_mask = pat.duplicated(subset=["person_key"], keep="first")
print(f"{len(pat):,} patient rows, {pat['person_key'].nunique():,} distinct people by phone+name+dob, {int(dup_mask.sum()):,} duplicate rows")
bad_dob = pat["dob"].eq("1900-01-01")
print("placeholder dob rows:", int(bad_dob.sum()))
pat_clean = pat[~dup_mask].copy()
checks.reconciles("patients de-duplicated", len(pat), len(pat_clean), int(dup_mask.sum()))
# map every duplicate id to its surviving id so bookings can be re-keyed to people, not to ids
id_map = pat.merge(pat_clean[["person_key", "patient_id"]].rename(columns={"patient_id": "person_id"}), on="person_key")[["patient_id", "person_id"]]
bookings = bookings.merge(id_map, on="patient_id", how="left", validate="many_to_one")
checks.check("every booking patient id maps to a person", bool(bookings["person_id"].notna().all()), int(bookings["person_id"].isna().sum()))

### The decision here is not technical

Merging two patient rows says two records are one human. For a growth analysis the cost of being wrong is a slightly wrong frequency figure. For a results system it would be a wrong report sent to the wrong person. Same rule, different stakes. Write the stakes in the decisions log beside the rule.

## 4 · DO: invoices and payments, two id families

In [ ]:
inv = pd.concat([pd.read_csv(p, dtype="string") for p in sorted((CONFIG["raw"] / "invoices").glob("*.csv.gz"))], ignore_index=True)
inv["amount_rs"] = inv["amount"].str.replace(",", "", regex=False).astype(float)
inv["invoice_date"] = pd.to_datetime(inv["invoice_date"])
pay = pd.concat([pd.read_csv(p, dtype={"notes_invoice_ref": "string", "settlement_id": "string"}) for p in sorted((CONFIG["raw"] / "payments").glob("gateway_*.csv.gz"))], ignore_index=True)
pay["amount_rs"] = pay["amount_paise"] / 100
pay["created_at"] = pd.to_datetime(pay["created_at_epoch"], unit="s") + pd.Timedelta(hours=5, minutes=30)
pay["invoice_id"] = loaders.normalise_invoice_ref(pay["notes_invoice_ref"])
print(f"invoices {len(inv):,} for Rs {inv['amount_rs'].sum():,.0f};  payment rows {len(pay):,} ({(pay.status=='captured').sum():,} captured, {(pay.status=='failed').sum():,} failed)")
checks.check("invoice ids unique", inv["invoice_id"].is_unique)

In [ ]:
captured = pay[pay["status"] == "captured"]
m = inv.merge(captured[["invoice_id", "payment_id", "amount_rs", "created_at"]].rename(columns={"amount_rs": "paid_rs"}), on="invoice_id", how="outer", indicator=True, validate="one_to_many")
print(m["_merge"].value_counts())
multi = m[m["_merge"] == "both"].groupby("invoice_id").size()
print("invoices with more than one captured payment:", int((multi > 1).sum()))

> **Interview angle.** Your LEFT join of invoices to payments grew the row count. Name three possible causes and the check that separates them.
>
> *What a good answer covers:* Retries (same invoice, a failed and a captured row): filter by status before joining. Genuine double charges (two captured rows): group by invoice and count captured. A many-to-many on a non-unique key such as amount and date: check key uniqueness on the right side with validate='one_to_many' or a groupby count before the join.

## 5 · SUM: write the clean tables and log every rule

In [ ]:
bookings.to_csv(CONFIG["clean"] / "bookings_clean.csv.gz", index=False, compression="gzip")
coll_clean.to_csv(CONFIG["clean"] / "collections_clean.csv.gz", index=False, compression="gzip")
pat_clean.to_csv(CONFIG["clean"] / "patients_clean.csv.gz", index=False, compression="gzip")
inv.to_csv(CONFIG["clean"] / "invoices_clean.csv.gz", index=False, compression="gzip")
pay.to_csv(CONFIG["clean"] / "payments_clean.csv.gz", index=False, compression="gzip")
print("headline after cleaning:", HEADLINE(bookings))
checks.summary()

## Before you move on

- List every rule you applied, in order, with the headline number before and after each. That list is `docs/decisions-log.md`; if it is not there yet, stop and write it.
- Which rule moved the headline most? That rule is on slide 5.
- Which rule would the finance controller challenge first?

## Short answers

1. **Rules:** Profile and retain source rows; normalize only safe city/reference formatting; exclude superseded records from active volume; separate failed from captured attempts; match exact then unique hyphen-stripped invoice IDs; keep unresolved matches and overcaptures separate; apply IST cutoffs.
2. **Active-volume rule:** Exclude 3,541 superseded H1 2026 reschedule rows. This is a stated cohort rule, not a hidden deletion.
3. **Likely Finance challenge:** Source completeness, reference identity, refunds/reversals, and missing independent bank/cash evidence. The bridge flags those limits and does not label balances as losses.

## A/B methods and the slide chart · Group 10

We first checked H1 scheduled volume, cleaned and profiled the data, and used Brief A/B as exploratory work. Group 10 selected Brief C as the board-facing finance question.

Active scheduled test lines grew 5%: 457,539 in H1 2026 versus 435,880 in H1 2025, against 18% estimated growth in the plan. A/B methods remain as supporting context; the Brief C bridge is the final analysis.

**Workflow:** profile and clean the booking data above, then run the H1 and Brief A/B analysis. The finance bridge has its own cleaning step in Notebook 02.

## CONFIG

In [ ]:
import sys, pathlib, time
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "notebooks" / "_scripts"))
import numpy as np
import pandas as pd
from kalpa import loaders, models, checks, logs
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
CONFIG = {
    "brief": "Brief C",
    "group": "10",
    "raw": ROOT / "data" / "raw",
    "clean": ROOT / "data" / "clean",
    "period_this": ("2026-01-01", "2026-07-01"),
    "period_last": ("2025-01-01", "2025-07-01"),
    "seed": 20261019,
    "workers": 2,             # Keep the full-pack load within Codespace memory.
}
CONFIG["clean"].mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(CONFIG["seed"])
print(pd.__version__, np.__version__)

bk = pd.read_csv(CONFIG["clean"] / "bookings_clean.csv.gz", parse_dates=["slot_at", "collected_at"], dtype={"booking_id": "string", "person_id": "string", "rescheduled_from": "string", "status_at_export": "string"})
is_last = bk["slot_at"].ge(pd.Timestamp(CONFIG["period_last"][0])) & bk["slot_at"].lt(pd.Timestamp(CONFIG["period_last"][1]))
is_this = bk["slot_at"].ge(pd.Timestamp(CONFIG["period_this"][0])) & bk["slot_at"].lt(pd.Timestamp(CONFIG["period_this"][1]))
bk["half"] = np.select([is_last, is_this], ["H1_2025", "H1_2026"], default="other")
bk = bk[bk["half"] != "other"]
print(bk.groupby("half").agg(bookings=("booking_id", "size"), tests=("n_tests", "sum"), people=("person_id", "nunique")))

## 1 · The tree (brief A)

Tests = people × visits per person × tests per visit. Growth in a product is close to the sum of growths in the factors (exactly so in logs). Compute each factor for both halves and check that they multiply back.

In [ ]:
def tree(df, by=None):
    g = df.groupby(([by] if by else []) + ["half"], observed=True).agg(tests=("n_tests", "sum"), visits=("booking_id", "size"), people=("person_id", "nunique"))
    g["visits_per_person"] = g["visits"] / g["people"]
    g["tests_per_visit"] = g["tests"] / g["visits"]
    return g

t = tree(bk)
growth = (t.xs("H1_2026") / t.xs("H1_2025") - 1) * 100
print(growth.round(2))
# log-decomposition: the branch growths sum to the total growth exactly
logs_ = np.log(t.xs("H1_2026") / t.xs("H1_2025"))
checks.check("branches multiply back to the total (log decomposition)", abs(logs_["people"] + logs_["visits_per_person"] + logs_["tests_per_visit"] - logs_["tests"]) < 1e-9,
             f"people {logs_['people']:+.4f} + visits/person {logs_['visits_per_person']:+.4f} + tests/visit {logs_['tests_per_visit']:+.4f} = {logs_['tests']:+.4f}")

In [ ]:
by_city = tree(bk, "city")
city_growth = ((by_city.xs("H1_2026", level="half") / by_city.xs("H1_2025", level="half") - 1) * 100).round(1)
city_growth[["tests", "people", "visits_per_person", "tests_per_visit"]].sort_values("tests")

> **Interview angle.** Revenue grew 10 percent and patients grew 4. Decompose the rest without a calculator, then say what you would check before presenting it.
>
> *What a good answer covers:* Roughly 6 points sit in visits per patient and revenue per visit together; log growth makes the split exact. Check that the patient count is a count of people and not of ids (duplicates), that both periods use the same definition of a visit, and that a system change did not move the grain.

## 2 · The ladder (brief B)

Confirm the drop, compare like with like, decompose, isolate, hypothesise. In code the ladder is a sequence of groupbys where each step changes one thing: the grain, the window, the segment.

In [ ]:
bk["quarter"] = bk["slot_at"].dt.to_period("Q").astype(str)
q = bk[bk["city"].isin(["Chennai", "Jaipur"])].groupby(["city", "quarter", "system"], observed=True).agg(bookings=("booking_id", "size"), tests=("n_tests", "sum")).reset_index()
pivot = q.pivot_table(index=["city", "quarter"], columns="system", values=["bookings", "tests"], aggfunc="sum", fill_value=0)
pivot

In [ ]:
# Rung 2, like with like: the same quarter a year earlier, per city, at booking grain, after the identity rule
yoy = bk[bk["city"].isin(["Chennai", "Jaipur"])].groupby(["city", "quarter"], observed=True)["booking_id"].size().unstack("quarter")
def pct_change(table, later, earlier, label):
    """Like with like means both periods must exist; say so when one does not, rather than fail quietly."""
    if later in table and earlier in table:
        print(((table[later] / table[earlier] - 1) * 100).round(1).rename(label))
    else:
        print(f"{label}: period {earlier if earlier not in table else later} is not in this pack (sample data?); nothing to compare yet")
pct_change(yoy, "2026Q2", "2025Q2", "Q2 2026 vs Q2 2025, bookings, %")

## 6 · SUM: the chart for slide 4

One chart, the claim's number against its comparison, with the denominator in the axis label. Save it to `slides/`.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 4.2))
g = city_growth["tests"].sort_values()
ax.barh(g.index, g.values, color=["#D63A6A" if v < 0 else "#4E38AB" for v in g.values])
ax.axvline(0, color="#1A1440", lw=1)
ax.set_xlabel("Active scheduled test lines, H1 2026 vs H1 2025 (denominator: H1 2025 scheduled lines)")
ax.set_title("Where the growth went, by city")
for i, v in enumerate(g.values): ax.text(v + (0.3 if v >= 0 else -0.3), i, f"{v:+.1f}%", va="center", ha="left" if v >= 0 else "right", fontsize=9)
plt.tight_layout()
(ROOT / "slides").mkdir(exist_ok=True)
fig.savefig(ROOT / "outputs" / "brief_a_h1_city_growth.png", dpi=200)
print("saved outputs/brief_a_h1_city_growth.png")
checks.summary()

## Regenerate the H1 and Brief A/B aggregate outputs

The analysis helpers write identifier-free tables under `outputs/`. The Jaipur drill-down is printed as supporting evidence; migration results remain descriptive, not causal.

In [ ]:
import subprocess
import sys

scripts = ROOT / "notebooks" / "_scripts"
subprocess.run([sys.executable, str(scripts / "analyze_brief_a_b_migration.py"), "--data-dir", str(ROOT / "data" / "clean"), "--output-dir", str(ROOT / "outputs")], cwd=ROOT, check=True)
subprocess.run([sys.executable, str(scripts / "analyze_brief_a_jaipur.py"), "--data-dir", str(ROOT / "data" / "clean")], cwd=ROOT, check=True)
print("Growth summary:")
print(pd.read_csv(ROOT / "outputs" / "brief_a_h1_overall.csv").to_string(index=False))